In [1]:
import numpy as np

# SVD studies

## Knob — Output correlation analysis

### First toy model: 5 observables driven by 3 knobs (generic)

In [2]:
# 1. Define the true underlying relationship (Response Matrix R)
# Rows = 5 Observables, Columns = 3 Knobs
R_true = np.array([
    [1.5,  1.4,  0.1],  # Obs 0: Strongly driven by Knob 0 & 1
    [-0.8, -0.7,  0.0], # Obs 1: Driven inversely by Knob 0 & 1
    [0.2,   0.2,  2.5], # Obs 2: Strongly driven by Knob 2
    [1.0,   0.9, -1.2], # Obs 3: Driven by all knobs
    [0.0,   0.0,  0.5]  # Obs 4: Weakly driven by Knob 2
])

In [3]:
# 2. Generate synthetic data (100 experiments/snapshots)
np.random.seed(42)
n_snapshots = 100

# Generate correlated inputs for Knob 0 and Knob 1, independent for Knob 2
knob_0 = np.random.normal(0, 1.0, n_snapshots)
knob_1 = knob_0 + np.random.normal(0, 0.1, n_snapshots) # Highly correlated to Knob 0
knob_2 = np.random.normal(0, 1.5, n_snapshots)          # Independent

# Combine into a Knob Matrix K (shape: 100 x 3)
K = np.column_stack((knob_0, knob_1, knob_2))

In [9]:
K.shape

(100, 3)

In [26]:
# 3. Calculate Observables Matrix Y (shape: 100 x 5) and add heavy noise
noise = np.random.normal(0, 0.5, size=(n_snapshots, 5))
Y = K @ R_true.T + noise

In [27]:
Y.shape

(100, 5)

In [ ]:
# 4. Perform SVD on the Observables to find dominant modes
# Center the data first (standard practice)
Y_centered = Y - np.mean(Y, axis=0)
U, S, VT = np.linalg.svd(Y_centered, full_matrices=False)

In [29]:
S

array([44.68774284, 33.69343193,  5.50422757,  4.60067906,  4.35649723])

In [30]:
# 5. Correlate SVD modes back to the original Knobs
# Project the original knobs onto the SVD space to see which knobs drive which modes
knob_correlation = np.corrcoef(K.T, U.T)[:3, 3:]

In [31]:
# Print results
print("Singular Values (Dominant patterns found):")
print(S)
print("\nCorrelation Matrix between Knobs (Rows) and SVD Observable Modes (Cols):")
print(np.round(knob_correlation, 2))


Singular Values (Dominant patterns found):
[44.68774284 33.69343193  5.50422757  4.60067906  4.35649723]

Correlation Matrix between Knobs (Rows) and SVD Observable Modes (Cols):
[[-0.28 -0.95 -0.01  0.   -0.01]
 [-0.28 -0.95 -0.    0.01 -0.01]
 [-0.99  0.09 -0.    0.    0.  ]]


In [ ]:
# Reconstruct the clean, noise-filtered Response Matrix (R_interpreted)
# We only use the first 2 columns/rows because the rest is noise
k_modes = 2 

U_clean = U[:, :k_modes]
S_clean = np.diag(S[:k_modes])
Vt_clean = VT[:k_modes, :]

# Calculate the pseudo-inverse of K to map directly from Knobs to Observables
# This gives us the empirical Response Matrix: Y_centered = K @ R^T
R_empirical = np.linalg.pinv(K) @ Y_centered

print("--- Noise-Filtered Response Matrix (Predicted by SVD) ---")
print("Rows = Observables (0 to 4), Columns = Knobs (0 to 2)\n")
print(np.round(R_empirical.T, 2))
print("\n--- True Response Matrix (Ground Truth) ---")
print(np.round(R_true, 2))


--- Noise-Filtered Response Matrix (Predicted by SVD) ---
Rows = Observables (0 to 4), Columns = Knobs (0 to 2)

[[ 0.8   1.97  0.06]
 [-1.09 -0.41  0.  ]
 [ 0.24  0.31  2.43]
 [ 0.53  1.39 -1.16]
 [-0.21  0.17  0.48]]

--- True Response Matrix (Ground Truth) ---
[[ 1.5  1.4  0.1]
 [-0.8 -0.7  0. ]
 [ 0.2  0.2  2.5]
 [ 1.   0.9 -1.2]
 [ 0.   0.   0.5]]


### Second toy model — Mechanical platform pushed by 3 motors, monitored by 5 sensors

In [37]:
# --- The Hidden Real System (The Black Box) ---
def physical_system(knobs):
    """Simulates a non-linear mechanical system with noise."""
    k0, k1, k2 = knobs
    # Hidden coupling: Knobs 0 and 1 affect sensors similarly; Knob 2 is independent
    obs0 = 2.0 * k0 + 1.8 * k1 + 0.1 * np.sin(k2)
    obs1 = -1.0 * k0 - 0.9 * k1 + 0.0
    obs2 = 0.5 * k0 + 0.5 * k1 + 3.0 * k2
    obs3 = 1.2 * k0 + 1.1 * k1 - 1.5 * k2
    obs4 = 0.0 * k0 + 0.0 * k1 + 0.8 * np.cos(k2)
    
    # Add small sensor measurement noise
    noise = np.random.normal(0, 0.02, size=5)
    return np.array([obs0, obs1, obs2, obs3, obs4]) + noise

In [38]:
# --- Step 1: Establish your baseline operating point ---
baseline_knobs = np.array([1.0, 1.0, 1.0])
baseline_observables = physical_system(baseline_knobs)

In [40]:
print(f"Baseline Knobs: {baseline_knobs}")
print(f"Baseline Observables: {np.round(baseline_observables, 2)}")

Baseline Knobs: [1. 1. 1.]
Baseline Observables: [ 3.92 -1.9   3.99  0.77  0.4 ]


In [41]:
# --- Step 2: Manually calculate the Jacobian Matrix (J) ---
# J_ij = d(Observable_i) / d(Knob_j)
delta_k = 0.01  # The small perturbation ("nudge") you apply to the system
n_observables = 5
n_knobs = 3

# Initialize an empty Jacobian matrix (5 rows for Observables, 3 columns for Knobs)
Jacobian = np.zeros((n_observables, n_knobs))

for j in range(n_knobs):
    nudged_knobs = baseline_knobs.copy()
    # Nudge ONLY knob 'j'
    nudged_knobs[j] += delta_k
    
    # Measure the new system output
    new_observables = physical_system(nudged_knobs)
    
    # Calculate finite difference slope: (Y_new - Y_baseline) / delta_K
    Jacobian[:, j] = (new_observables - baseline_observables) / delta_k

print("--- Step 2: Your Manually Calculated Jacobian Matrix ---")
print(np.round(Jacobian, 3))
print("\n" + "="*50 + "\n")

--- Step 2: Your Manually Calculated Jacobian Matrix ---
[[-4.971 -2.969 -5.531]
 [-0.344 -0.485 -4.182]
 [ 2.934  5.141  2.979]
 [ 1.065  5.921  1.105]
 [ 5.396  3.069  0.957]]




In [229]:
# --- Step 3: Run SVD on your custom Jacobian ---
# We analyze J directly to find structural patterns between inputs and outputs
U, S, VT = np.linalg.svd(Jacobian, full_matrices=False)

print("--- Step 3: SVD Structural Analysis ---")
print(f"Singular Values: {np.round(S, 3)}")
print("Notice how the 3rd value is near-zero? The system has only 2 true dimensions of control!")

print("\nRight Singular Vectors (Vᵀ Matrix - How Knobs group together):")
print(np.round(VT, 2))
print("Row 0 shows Knob 0 and 1 are mathematically locked together in Mode 1.")
print("Row 1 shows Knob 2 operates entirely independently in Mode 2.")


--- Step 3: SVD Structural Analysis ---
Singular Values: [78.919  7.052  3.432]
Notice how the 3rd value is near-zero? The system has only 2 true dimensions of control!

Right Singular Vectors (Vᵀ Matrix - How Knobs group together):
[[-0.56 -0.57 -0.61]
 [ 0.03  0.72 -0.69]
 [ 0.83 -0.4  -0.39]]
Row 0 shows Knob 0 and 1 are mathematically locked together in Mode 1.
Row 1 shows Knob 2 operates entirely independently in Mode 2.


In [230]:
# Desired outputs

y_star = baseline_observables + np.array([0.1, -0.1, 0.2, 0.0, -0.05])  # Example target change in observables

r = 1 # rank of reconstruction

R = VT.T[:, :r] @ np.diag(1 / S[:r]) @ U.T[:r, :]  # Reconstruct the Response Matrix from SVD components

x_star = R @ y_star  # Calculate the required knob settings to achieve y_star

x_star

array([0.04332011, 0.04387075, 0.04718645])

In [ ]:
new_knobs = baseline_knobs + x_star  # Adjust knobs from baseline to achieve target observables
y_actual = physical_system(new_knobs)

print(y_actual)
print(y_star)
print(np.abs(y_actual - y_star).sum())  # Check how close we got to the target

[ 4.03507412 -1.99391472  4.18924256  0.8054193   0.39209408]
[ 4.02217091 -2.00121322  4.18583186  0.76972571  0.34617905]
0.10522100800124107


In [228]:
# --- Step 2: Manually calculate the Jacobian Matrix (J) ---
# J_ij = d(Observable_i) / d(Knob_j)
delta_k = 0.01  # The small perturbation ("nudge") you apply to the system
n_observables = 5
n_knobs = 3

# Initialize an empty Jacobian matrix (5 rows for Observables, 3 columns for Knobs)
Jacobian = np.zeros((n_observables, n_knobs))

for j in range(n_knobs):
    nudged_knobs = new_knobs.copy()
    # Nudge ONLY knob 'j'
    nudged_knobs[j] += delta_k
    
    # Measure the new system output
    new_observables = physical_system(nudged_knobs)
    
    # Calculate finite difference slope: (Y_new - Y_baseline) / delta_K
    Jacobian[:, j] = (new_observables - baseline_observables) / delta_k

print("--- Step 2: Your Manually Calculated Jacobian Matrix ---")
print(np.round(Jacobian, 3))
print("\n" + "="*50 + "\n")

--- Step 2: Your Manually Calculated Jacobian Matrix ---
[[ 25.515  30.212  26.37 ]
 [-15.865 -13.113 -12.195]
 [ 30.354  27.808  36.957]
 [ 11.08   12.494  10.697]
 [ -0.51    1.48   -1.382]]


